# NB11 — Mechanisms

NB05/NB10 established two findings; this notebook probes *why* they hold:

1. **Amplification** — MTS → `log_brd` is positive (β≈+2.03, full sample). Is it
   systemic across the panel, or driven by a handful of outlier states?
2. **Africa concentration** — the Africa marginal effect on `log_brd` (β≈+4.95) is
   robust across MTS variants. Is it genuinely geographic, or regime type
   (polyarchy) wearing a geographic costume?

Estimator: the NB05 two-way FE specification throughout, now imported from
`src.stats_panel` (refactored out of NB10 — `run_panel_fe`, `marginal_effect`,
`run_panel_fe_interactions`, `joint_wald_interactions`). No estimator settings
changed.

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Leave-one-country-out jackknife (is amplification systemic?) |
| 2 | Africa-focused jackknife (is the Africa effect a few states?) |
| 3 | Regime-type moderation (polyarchy) |
| 4 | Region vs polyarchy: the confounding test |
| 5 | Conflict-type decomposition (why substitution failed) |
| 5b | Lagged MTS (reverse-causality robustness) |
| 6 | Robustness across MTS variants |
| 7 | Sanity checks |
| 8 | Headline findings |

**Note on `AbsorbingEffectWarning`:** region dummies are time-invariant, so their
main effects are absorbed by EntityEffects and removed via `drop_absorbed=True` —
expected and correct (NB10 Sections 3/5c behave identically). The warnings are
silenced around region-interaction fits to keep jackknife loops readable.

## Section 0 — Setup

Load `rq1_panel_grouped.parquet`; derive `log_brd` only if the checkpoint lacks it;
rebuild region dummies from whichever source the checkpoint provides.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from src.io_utils import load_checkpoint
from src.config import CLEAN_DIR, FIGURES_DIR, TABLES_DIR
from src.stats_panel import (run_panel_fe, marginal_effect,
                             run_panel_fe_interactions, joint_wald_interactions)

pd.set_option("display.width", 160)

FIG_DIR = FIGURES_DIR / "nb11"
TBL_DIR = TABLES_DIR / "nb11"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

rq1 = load_checkpoint(CLEAN_DIR / "rq1_panel_grouped.parquet")
assert len(rq1) == 6912, f"expected 6,912 rows, got {len(rq1):,}"

# log_brd: NB10 derived it before saving the grouped checkpoint, so it should be
# present. If a future re-run of NB10 changes that, replicate the NB10 Section 0
# derivation exactly.
if "log_brd" in rq1.columns:
    print("log_brd present in checkpoint — no derivation needed "
          f"({int((rq1['log_brd'] > 0).sum()):,} non-zero country-years)")
else:
    master = load_checkpoint(CLEAN_DIR / "master_panel.parquet")
    _brd = (master.loc[(master["year"] >= 1989) & (master["year"] <= 2024),
                       ["iso3", "year", "brd_deaths_best"]]
                  .drop_duplicates(["iso3", "year"]))
    rq1 = rq1.merge(_brd, on=["iso3", "year"], how="left")
    assert len(rq1) == 6912, "brd merge fan-out"
    rq1["log_brd"] = np.log1p(rq1["brd_deaths_best"].fillna(0))
    rq1 = rq1.sort_values(["iso3", "year"]).reset_index(drop=True)
    rq1["log_brd_lag1"] = rq1.groupby("iso3")["log_brd"].shift(1)
    print("log_brd DERIVED from master_panel (checkpoint lacked it)")

CONTROLS = ["log_gdp", "log_pop", "vdem_v2x_polyarchy"]
PRIMARY_MTS = "mts_pca_3feat"
MTS_VARIANTS = ["mts_pca_3feat", "mts_milex", "mts_tiv"]
REGION_REF = "Africa"

# Region dummies: use checkpoint region_* columns if present, else rebuild
existing_dummies = sorted(c for c in rq1.columns if c.startswith("region_"))
if existing_dummies:
    region_dummy = {c.removeprefix("region_"): c for c in existing_dummies}
    print(f"Region dummies FOUND in checkpoint: {existing_dummies}")
else:
    def _sanitize(name):
        return "".join(ch for ch in name if ch.isalnum())
    region_dummy = {}
    for r in sorted(rq1["region"].unique()):
        if r == REGION_REF:
            continue
        col = f"region_{_sanitize(r)}"
        rq1[col] = (rq1["region"] == r).astype(int)
        region_dummy[r] = col
    print(f"Region dummies REBUILT from `region` column (absent from checkpoint): "
          f"{list(region_dummy.values())}")

print(f"\nShape: {rq1.shape} | years {rq1['year'].min()}–{rq1['year'].max()} | "
      f"{rq1['iso3'].nunique()} countries")
print(f"Region counts:\n{rq1['region'].value_counts().to_string()}")
print("\n[Section 0] Setup complete — panel loaded, dummies ready.")

[checkpoint] loaded ← rq1_panel_grouped.parquet  (6,912 rows)
log_brd present in checkpoint — no derivation needed (1,094 non-zero country-years)
Region dummies REBUILT from `region` column (absent from checkpoint): ['region_Americas', 'region_Asia', 'region_Europe', 'region_MiddleEast', 'region_Oceania', 'region_PostSoviet']

Shape: (6912, 36) | years 1989–2024 | 192 countries
Region counts:
region
Africa         1764
Europe         1476
Americas       1152
Asia            900
Middle East     720
Oceania         468
Post-Soviet     432

[Section 0] Setup complete — panel loaded, dummies ready.


## Section 1 — Leave-One-Country-Out Jackknife

Is the amplification effect (MTS → `log_brd`, plain NB05 spec, no interaction)
systemic or outlier-driven? Refit dropping each country in turn (~190 linear FE
fits), plus one joint fit dropping the top-5 countries by mean MTS.

In [2]:
full_res, _ = run_panel_fe(rq1, "log_brd", PRIMARY_MTS, CONTROLS)
full_b = full_res.params[PRIMARY_MTS]
full_p = full_res.pvalues[PRIMARY_MTS]
print(f"Full-sample: β={full_b:+.4f} (SE {full_res.std_errors[PRIMARY_MTS]:.4f}, "
      f"p={full_p:.4f}, N={int(full_res.nobs)})\n")

loo_rows, loo_failures = [], []
for iso3 in sorted(rq1["iso3"].unique()):
    try:
        res, _ = run_panel_fe(rq1[rq1["iso3"] != iso3], "log_brd",
                              PRIMARY_MTS, CONTROLS)
        loo_rows.append({
            "iso3": iso3,
            "β":  round(res.params[PRIMARY_MTS], 4),
            "SE":     round(res.std_errors[PRIMARY_MTS], 4),
            "p":      round(res.pvalues[PRIMARY_MTS], 4),
            "N":      int(res.nobs),
        })
    except Exception as e:
        loo_failures.append({"iso3": iso3, "error": str(e)[:120]})

loo_df = pd.DataFrame(loo_rows)
print(f"LOO fits: {len(loo_df)} successful, {len(loo_failures)} failed")
if loo_failures:
    print("Failed fits:")
    for f in loo_failures:
        print(f"  {f['iso3']}: {f['error']}")

# Joint drop of the top-5 countries by mean MTS
top5 = rq1.groupby("iso3")[PRIMARY_MTS].mean().nlargest(5)
print(f"\nTop-5 countries by mean {PRIMARY_MTS}: "
      + ", ".join(f"{i} ({v:.2f})" for i, v in top5.items()))
res5, _ = run_panel_fe(rq1[~rq1["iso3"].isin(top5.index)], "log_brd",
                       PRIMARY_MTS, CONTROLS)
drop5_b, drop5_p = res5.params[PRIMARY_MTS], res5.pvalues[PRIMARY_MTS]
print(f"Drop-top-5 jointly: β={drop5_b:+.4f} (p={drop5_p:.4f}, N={int(res5.nobs)})")

min_row = loo_df.loc[loo_df["β"].idxmin()]
max_row = loo_df.loc[loo_df["β"].idxmax()]
print(f"\nLOO β range: [{min_row['β']:+.4f} (drop {min_row['iso3']}), "
      f"{max_row['β']:+.4f} (drop {max_row['iso3']})]")

breakers = loo_df[(loo_df["β"] <= 0) | (loo_df["p"] >= 0.05)]
drop5_ok = (drop5_b > 0) and (drop5_p < 0.05)
if len(breakers) == 0:
    sec1_verdict = ("SYSTEMIC — every LOO β stays positive with p<0.05; drop-top-5 "
                    + ("also survives" if drop5_ok else
                       f"does NOT survive (β={drop5_b:+.3f}, p={drop5_p:.3f})"))
else:
    sec1_verdict = ("NOT systemic — broken by: "
                    + ", ".join(f"{r['iso3']} (β={r['β']:+.3f}, p={r['p']:.3f})"
                                for _, r in breakers.iterrows()))
print(f"\nVerdict: {sec1_verdict}")

loo_out = pd.concat([
    loo_df,
    pd.DataFrame([{"iso3": "(drop top-5 jointly: " + ", ".join(top5.index) + ")",
                   "β": round(drop5_b, 4), "SE": round(res5.std_errors[PRIMARY_MTS], 4),
                   "p": round(drop5_p, 4), "N": int(res5.nobs)}]),
], ignore_index=True)
loo_out.to_csv(TBL_DIR / "section1_loo_jackknife.csv", index=False)

# Figure: LOO β distribution
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(loo_df["β"], bins=40, color="steelblue", edgecolor="white")
ax.axvline(full_b, color="indianred", lw=1.4,
           label=f"full-sample β = {full_b:+.3f}")
for row, lbl in [(min_row, "min"), (max_row, "max")]:
    ax.axvline(row["β"], color="gray", lw=0.8, ls=":")
    ax.text(row["β"], ax.get_ylim()[1] * 0.92, f" {lbl}: drop {row['iso3']}",
            rotation=90, fontsize=8, ha="left" if lbl == "max" else "right",
            va="top")
ax.set_xlabel(f"β ({PRIMARY_MTS}) on log_brd, leave-one-country-out", fontsize=9)
ax.set_ylabel("countries", fontsize=9)
ax.set_title("Leave-one-country-out distribution of the amplification effect",
             fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_loo_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"\n[Section 1] LOO jackknife done ({len(loo_df)} fits) — saved → "
      f"section1_loo_jackknife.csv + fig1_loo_distribution.png")

Full-sample: β=+2.0297 (SE 0.5915, p=0.0006, N=4896)



LOO fits: 192 successful, 0 failed

Top-5 countries by mean mts_pca_3feat: USA (0.51), SAU (0.46), IND (0.46), CHN (0.46), GBR (0.43)
Drop-top-5 jointly: β=+2.0255 (p=0.0006, N=4721)

LOO β range: [+1.8185 (drop LKA), +2.1574 (drop KWT)]

Verdict: SYSTEMIC — every LOO β stays positive with p<0.05; drop-top-5 also survives



[Section 1] LOO jackknife done (192 fits) — saved → section1_loo_jackknife.csv + fig1_loo_distribution.png


## Section 2 — Africa-Focused Jackknife

Is the Africa marginal effect (NB10 Section 3 model: MTS × region dummies, Africa =
reference, so Africa marginal = β_mts) carried by a few African states? Refit
dropping each African country that has any non-zero `log_brd`, plus the top-3
contributors jointly.

Note that the Africa effect is estimated on the **data-complete subset** of African states: several high-conflict states (e.g. SOM) lack usable MTS/controls and are invisible to the regression — coverage is a *caveat*, not an inflation risk, since their conflict-years contribute nothing to the estimate.

In [3]:
afr_nz = rq1[(rq1["region"] == "Africa") & (rq1["log_brd"] > 0)]
afr_counts = afr_nz["iso3"].value_counts()
print(f"African countries with non-zero log_brd: {len(afr_counts)}")
print(afr_counts.to_string())

sec2_rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    res0, int_cols0 = run_panel_fe_interactions(rq1, "log_brd", PRIMARY_MTS,
                                                CONTROLS, region_dummy)
    sec2_rows.append({"drop": "(none)",
                      "africa_β": round(res0.params[PRIMARY_MTS], 4),
                      "SE": round(res0.std_errors[PRIMARY_MTS], 4),
                      "p": round(res0.pvalues[PRIMARY_MTS], 4),
                      "N": int(res0.nobs)})

    sec2_failures = []
    for iso3 in afr_counts.index:
        try:
            res, _ = run_panel_fe_interactions(rq1[rq1["iso3"] != iso3], "log_brd",
                                               PRIMARY_MTS, CONTROLS, region_dummy)
            sec2_rows.append({"drop": iso3,
                              "africa_β": round(res.params[PRIMARY_MTS], 4),
                              "SE": round(res.std_errors[PRIMARY_MTS], 4),
                              "p": round(res.pvalues[PRIMARY_MTS], 4),
                              "N": int(res.nobs)})
        except Exception as e:
            sec2_failures.append({"iso3": iso3, "error": str(e)[:120]})

    top3_afr = list(afr_counts.index[:3])
    res3, _ = run_panel_fe_interactions(rq1[~rq1["iso3"].isin(top3_afr)], "log_brd",
                                        PRIMARY_MTS, CONTROLS, region_dummy)
    sec2_rows.append({"drop": "top-3 jointly (" + ", ".join(top3_afr) + ")",
                      "africa_β": round(res3.params[PRIMARY_MTS], 4),
                      "SE": round(res3.std_errors[PRIMARY_MTS], 4),
                      "p": round(res3.pvalues[PRIMARY_MTS], 4),
                      "N": int(res3.nobs)})

sec2_df = pd.DataFrame(sec2_rows)
print(f"\nRefits: {len(sec2_df) - 1} ({len(afr_counts)} single drops + 1 joint), "
      f"{len(sec2_failures)} failures")
if sec2_failures:
    for f in sec2_failures:
        print(f"  FAILED {f['iso3']}: {f['error']}")
# Sample-coverage flag: a drop that returns IDENTICAL (β, N) to the full
# sample was never in the estimation sample (missing MTS or controls)
base_row = sec2_df.loc[sec2_df["drop"] == "(none)"].iloc[0]
not_in_sample = [
    r["drop"] for _, r in sec2_df.iterrows()
    if r["drop"] in set(afr_counts.index)
    and r["africa_β"] == base_row["africa_β"] and r["N"] == base_row["N"]
]
sec2_df["in_sample"] = ~sec2_df["drop"].isin(not_in_sample)
print(f"\nCountries with non-zero log_brd but NOT in the estimation sample "
      f"(no usable MTS/controls): {not_in_sample} — their conflict-years "
      f"contribute nothing to the estimate.")

print()
print(sec2_df.to_string(index=False))

sec2_breakers = sec2_df[(sec2_df["drop"] != "(none)")
                        & ((sec2_df["africa_β"] <= 0) | (sec2_df["p"] >= 0.10))]
if len(sec2_breakers) == 0:
    sec2_verdict = ("BROAD — Africa β stays positive with p<0.10 in every "
                    "single-country drop and the joint top-3 drop")
else:
    base_b = sec2_df.loc[sec2_df["drop"] == "(none)", "africa_β"].iloc[0]
    sec2_verdict = ("NOT broad — broken by: "
                    + "; ".join(f"{r['drop']} (β={r['africa_β']:+.3f}, "
                                f"p={r['p']:.3f}, Δ={r['africa_β'] - base_b:+.3f})"
                                for _, r in sec2_breakers.iterrows()))
print(f"\nVerdict: {sec2_verdict}")

sec2_df.to_csv(TBL_DIR / "section2_africa_jackknife.csv", index=False)
print(f"\n[Section 2] Africa jackknife done — saved → "
      f"{TBL_DIR / 'section2_africa_jackknife.csv'}")

African countries with non-zero log_brd: 34
iso3
SDN    35
ETH    33
SOM    29
UGA    28
TCD    27
AGO    24
BDI    24
COD    23
RWA    22
MLI    19
CAF    18
NER    17
NGA    16
MOZ    14
SLE    11
SSD    11
CMR    10
KEN    10
SEN    10
BFA     7
COG     6
LBR     6
DJI     5
CIV     4
BEN     3
ERI     3
TGO     3
COM     2
GIN     2
GNB     2
LSO     1
MRT     1
TZA     1
ZAF     1



Refits: 35 (34 single drops + 1 joint), 0 failures

Countries with non-zero log_brd but NOT in the estimation sample (no usable MTS/controls): ['SOM', 'ERI', 'COM'] — their conflict-years contribute nothing to the estimate.

                         drop  africa_β     SE      p    N  in_sample
                       (none)    4.9485 1.3254 0.0002 4896       True
                          SDN    5.1331 1.4229 0.0003 4869       True
                          ETH    5.1974 1.3574 0.0001 4861       True
                          SOM    4.9485 1.3254 0.0002 4896      False
                          UGA    5.3099 1.3305 0.0001 4861       True
                          TCD    5.2971 1.2522 0.0000 4863       True
                          AGO    4.8492 1.5361 0.0016 4861       True
                          BDI    4.9246 1.3214 0.0002 4864       True
                          COD    4.8060 1.3467 0.0004 4865       True
                          RWA    5.1060 1.3570 0.0002 4861       True
    

## Section 3 — Regime-Type Moderation (Polyarchy)

MTS × polyarchy interaction for `part_n_minor` and `log_brd`. Because polyarchy is
the moderator here, the controls are reduced to `log_gdp` + `log_pop` — the
polyarchy main effect enters via the interaction machinery (no duplicate RHS term;
asserted). Marginal MTS effects evaluated at polyarchy = 0.2 / 0.5 / 0.8 with
delta-method SEs; dummy robustness with `is_democracy = (polyarchy ≥ 0.5)`.

In [4]:
POLY_CONTROLS = ["log_gdp", "log_pop"]
SEC3_OUTCOMES = ["part_n_minor", "log_brd"]
POLY = "vdem_v2x_polyarchy"


def marginal_at(res, mts, int_col, q):
    """β_mts + q·β_int with SE = sqrt(V[mts] + q²·V[int] + 2q·Cov)."""
    b = res.params[mts] + q * res.params[int_col]
    V = res.cov
    se = float(np.sqrt(V.loc[mts, mts] + q**2 * V.loc[int_col, int_col]
                       + 2 * q * V.loc[mts, int_col]))
    p = 2 * (1 - stats.norm.cdf(abs(b / se)))
    return b, se, p


# is_democracy: NaN polyarchy must stay NaN (not silently become 0) so those
# rows drop out of the fit exactly as they do in the continuous spec
rq1["is_democracy"] = np.where(rq1[POLY].isna(), np.nan,
                               (rq1[POLY] >= 0.5).astype(float))

sec3_rows = []
sec3_results = {}
sec3_reading = {}
for outcome in SEC3_OUTCOMES:
    res, int_col = run_panel_fe(rq1, outcome, PRIMARY_MTS, POLY_CONTROLS,
                                interaction=POLY)
    names = list(res.params.index)
    assert len(names) == len(set(names)), f"duplicate RHS terms: {names}"
    sec3_results[outcome] = (res, int_col)
    b_int, p_int = res.params[int_col], res.pvalues[int_col]

    q_stats = {}
    for q in (0.2, 0.5, 0.8):
        b, se, p = marginal_at(res, PRIMARY_MTS, int_col, q)
        q_stats[q] = (b, se, p)
        sec3_rows.append({"spec": "continuous", "Outcome": outcome,
                          "quantity": f"marginal β at polyarchy={q}",
                          "β": round(b, 4), "SE": round(se, 4), "p": round(p, 4),
                          "N": int(res.nobs)})
    sec3_rows.append({"spec": "continuous", "Outcome": outcome,
                      "quantity": "mts × polyarchy interaction",
                      "β": round(b_int, 4), "SE": round(res.std_errors[int_col], 4),
                      "p": round(p_int, 4), "N": int(res.nobs)})

    b02, _, p02 = q_stats[0.2]
    b08, _, p08 = q_stats[0.8]
    if (b02 > 0 and p02 < 0.05) and (p_int < 0.10 and b_int < 0):
        sec3_reading[outcome] = "amplification concentrated in autocracies"
    elif p_int < 0.10 and b_int > 0:
        sec3_reading[outcome] = "amplification stronger in democracies"
    else:
        sec3_reading[outcome] = "no significant regime-type moderation"
    print(f"{outcome}: β@0.2={b02:+.3f} (p={p02:.3f}), "
          f"β@0.5={q_stats[0.5][0]:+.3f} (p={q_stats[0.5][2]:.3f}), "
          f"β@0.8={b08:+.3f} (p={p08:.3f}), interaction p={p_int:.3f}")
    print(f"  → {sec3_reading[outcome]}")

    # Dummy robustness
    res_d, ic_d = run_panel_fe(rq1, outcome, PRIMARY_MTS, POLY_CONTROLS,
                               interaction="is_democracy")
    me_d, me_se_d, me_p_d = marginal_effect(res_d, PRIMARY_MTS, ic_d)
    sec3_rows += [
        {"spec": "dummy", "Outcome": outcome, "quantity": "β autocracy (is_dem=0)",
         "β": round(res_d.params[PRIMARY_MTS], 4),
         "SE": round(res_d.std_errors[PRIMARY_MTS], 4),
         "p": round(res_d.pvalues[PRIMARY_MTS], 4), "N": int(res_d.nobs)},
        {"spec": "dummy", "Outcome": outcome, "quantity": "β democracy (is_dem=1)",
         "β": round(me_d, 4), "SE": round(me_se_d, 4), "p": round(me_p_d, 4),
         "N": int(res_d.nobs)},
        {"spec": "dummy", "Outcome": outcome, "quantity": "mts × is_democracy",
         "β": round(res_d.params[ic_d], 4), "SE": round(res_d.std_errors[ic_d], 4),
         "p": round(res_d.pvalues[ic_d], 4), "N": int(res_d.nobs)},
    ]
    print(f"  dummy check: β_autocracy={res_d.params[PRIMARY_MTS]:+.3f} "
          f"(p={res_d.pvalues[PRIMARY_MTS]:.3f}), β_democracy={me_d:+.3f} "
          f"(p={me_p_d:.3f}), interaction p={res_d.pvalues[ic_d]:.3f}\n")

sec3_df = pd.DataFrame(sec3_rows)
sec3_df.to_csv(TBL_DIR / "section3_polyarchy_interaction.csv", index=False)
print(f"[Section 3] Polyarchy moderation fit for {len(SEC3_OUTCOMES)} outcomes — "
      f"saved → {TBL_DIR / 'section3_polyarchy_interaction.csv'}")

part_n_minor: β@0.2=+1.465 (p=0.111), β@0.5=+1.697 (p=0.009), β@0.8=+1.930 (p=0.108), interaction p=0.784
  → no significant regime-type moderation
  dummy check: β_autocracy=+1.204 (p=0.143), β_democracy=+2.380 (p=0.026), interaction p=0.397



log_brd: β@0.2=+2.274 (p=0.003), β@0.5=+1.988 (p=0.001), β@0.8=+1.701 (p=0.012), interaction p=0.497
  → no significant regime-type moderation
  dummy check: β_autocracy=+2.161 (p=0.002), β_democracy=+1.853 (p=0.007), interaction p=0.687

[Section 3] Polyarchy moderation fit for 2 outcomes — saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb11\section3_polyarchy_interaction.csv


In [5]:
# ── Fig 2: marginal MTS effect vs polyarchy (line + 95% CI band) ──────────────
q_grid = np.linspace(0, 1, 101)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
for ax, outcome in zip(axes, SEC3_OUTCOMES):
    res, int_col = sec3_results[outcome]
    me = np.array([marginal_at(res, PRIMARY_MTS, int_col, q)[0] for q in q_grid])
    se = np.array([marginal_at(res, PRIMARY_MTS, int_col, q)[1] for q in q_grid])
    ax.plot(q_grid, me, color="steelblue", lw=1.6)
    ax.fill_between(q_grid, me - 1.96 * se, me + 1.96 * se,
                    color="steelblue", alpha=0.2)
    ax.axhline(0, color="gray", linestyle="--", lw=0.8)
    p_int = res.pvalues[int_col]
    ax.set_title(f"{outcome}  (interaction p={p_int:.3f})", fontsize=10)
    ax.set_xlabel("V-Dem polyarchy", fontsize=9)
    ax.set_ylabel("marginal β (MTS)", fontsize=9)
fig.suptitle("Marginal MTS effect by regime type — two-way FE, clustered SE", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig2_marginal_by_polyarchy.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[Section 3] Figure saved → {FIG_DIR / 'fig2_marginal_by_polyarchy.png'}")

[Section 3] Figure saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\figures\nb11\fig2_marginal_by_polyarchy.png


## Section 4 — Region vs Polyarchy: The Confounding Test

Three nested models per outcome (controls = `log_gdp` + `log_pop`; polyarchy enters
M1 as a plain control, M2/M3 as a moderator):

- **M1** — MTS × region dummies only
- **M2** — MTS × polyarchy only
- **M3** — both interaction sets in one model (`run_panel_fe_interactions` just
  multiplies `sub[mts] * sub[col]`, so passing continuous polyarchy alongside the
  0/1 region dummies is valid — verified in `src/stats_panel.py`)

If the Africa marginal effect collapses from M1 → M3, "Africa" was regime type in
disguise; if it survives, the effect is geography/conflict-system specific.

In [6]:
CONTROLS_M = ["log_gdp", "log_pop"]
SEC4_OUTCOMES = ["log_brd", "part_n_minor"]
mods_m3 = dict(region_dummy)
mods_m3["polyarchy"] = POLY

sec4_rows = []
sec4_verdicts = {}
sec4_models = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for outcome in SEC4_OUTCOMES:
        m1, m1_ints = run_panel_fe_interactions(rq1, outcome, PRIMARY_MTS,
                                                CONTROLS_M + [POLY], region_dummy)
        m2, m2_int = run_panel_fe(rq1, outcome, PRIMARY_MTS, CONTROLS_M,
                                  interaction=POLY)
        m3, m3_ints = run_panel_fe_interactions(rq1, outcome, PRIMARY_MTS,
                                                CONTROLS_M, mods_m3)
        sec4_models[outcome] = (m1, m2, m3, m3_ints)

        a1, a1p = m1.params[PRIMARY_MTS], m1.pvalues[PRIMARY_MTS]
        a3, a3p = m3.params[PRIMARY_MTS], m3.pvalues[PRIMARY_MTS]
        p2, p2p = m2.params[m2_int], m2.pvalues[m2_int]
        p3col = m3_ints["polyarchy"]
        p3, p3p = m3.params[p3col], m3.pvalues[p3col]

        shrink_pct = 100 * (a1 - a3) / a1 if a1 != 0 else np.nan
        sec4_rows += [
            {"Outcome": outcome, "quantity": "Africa marginal (β_mts)",
             "M1": round(a1, 4), "p_M1": round(a1p, 4),
             "M3": round(a3, 4), "p_M3": round(a3p, 4),
             "shrink_M1→M3_%": round(shrink_pct, 1)},
            {"Outcome": outcome, "quantity": "mts × polyarchy",
             "M1": np.nan, "p_M1": np.nan,
             "M3": round(p3, 4), "p_M3": round(p3p, 4),
             "shrink_M1→M3_%": np.nan,
             "M2": round(p2, 4), "p_M2": round(p2p, 4)},
        ]

        print(f"--- {outcome} ---")
        print(f"  Africa marginal:  M1 β={a1:+.4f} (p={a1p:.4f})  →  "
              f"M3 β={a3:+.4f} (p={a3p:.4f})   shrink={shrink_pct:+.1f}%")
        print(f"  mts × polyarchy:  M2 β={p2:+.4f} (p={p2p:.4f})  →  "
              f"M3 β={p3:+.4f} (p={p3p:.4f})")

        if shrink_pct > 50 and a3p >= 0.10:
            v = ("region effect largely ABSORBED by regime type — polyarchy is "
                 "the mechanism candidate")
        elif shrink_pct < 25 and a3p < 0.10:
            v = "region effect SURVIVES regime controls — geography/conflict-system specific"
        else:
            v = (f"PARTIAL absorption — Africa β {a1:+.3f} → {a3:+.3f} "
                 f"({shrink_pct:+.1f}%), p {a1p:.3f} → {a3p:.3f}")
        sec4_verdicts[outcome] = v
        print(f"  Verdict: {v}\n")

sec4_df = pd.DataFrame(sec4_rows)
sec4_df.to_csv(TBL_DIR / "section4_region_vs_polyarchy.csv", index=False)
print(f"[Section 4] Nested M1/M2/M3 comparison done — saved → "
      f"{TBL_DIR / 'section4_region_vs_polyarchy.csv'}")

--- log_brd ---
  Africa marginal:  M1 β=+4.9485 (p=0.0002)  →  M3 β=+4.5340 (p=0.0006)   shrink=+8.4%
  mts × polyarchy:  M2 β=-0.9550 (p=0.4974)  →  M3 β=+1.1564 (p=0.6130)
  Verdict: region effect SURVIVES regime controls — geography/conflict-system specific



--- part_n_minor ---
  Africa marginal:  M1 β=+4.8774 (p=0.0073)  →  M3 β=+3.9380 (p=0.0544)   shrink=+19.3%
  mts × polyarchy:  M2 β=+0.7752 (p=0.7842)  →  M3 β=+2.6185 (p=0.5069)
  Verdict: region effect SURVIVES regime controls — geography/conflict-system specific

[Section 4] Nested M1/M2/M3 comparison done — saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb11\section4_region_vs_polyarchy.csv


## Section 5 — Conflict-Type Decomposition (Why Substitution Failed)

`part_n_territorial = (part_n_war + part_n_minor) − part_n_extraterritorial`
(clipped at 0; clip count reported). The substitution hypothesis needs territorial
participation DOWN and extraterritorial UP as MTS rises; both β's are estimated
with the plain NB05 spec. Lags are built for **both** derived outcomes so the two
specifications stay symmetric (the checkpoint carries no `part_*_lag1` columns).

In [7]:
raw_terr = rq1["part_n_war"] + rq1["part_n_minor"] - rq1["part_n_extraterritorial"]
n_clip = int((raw_terr < 0).sum())
print(f"Rows clipped at 0 in part_n_territorial derivation: {n_clip}")
if n_clip > 0:
    bad = rq1.loc[raw_terr < 0, ["iso3", "year", "part_n_war", "part_n_minor",
                                 "part_n_extraterritorial"]].head(10)
    print(bad.to_string(index=False))

rq1["part_n_territorial"] = raw_terr.clip(lower=0)
rq1 = rq1.sort_values(["iso3", "year"]).reset_index(drop=True)
# symmetric lagged DVs for a like-for-like comparison of the two margins
rq1["part_n_territorial_lag1"] = rq1.groupby("iso3")["part_n_territorial"].shift(1)
rq1["part_n_extraterritorial_lag1"] = (
    rq1.groupby("iso3")["part_n_extraterritorial"].shift(1))

SEC5_OUTCOMES = ["part_n_territorial", "part_n_extraterritorial"]
sec5_rows = []
sec5_stats = {}
for outcome in SEC5_OUTCOMES:
    res, _ = run_panel_fe(rq1, outcome, PRIMARY_MTS, CONTROLS)
    b, p = res.params[PRIMARY_MTS], res.pvalues[PRIMARY_MTS]
    sec5_stats[outcome] = (b, p)
    sec5_rows.append({"Outcome": outcome, "β_mts": round(b, 4),
                      "SE": round(res.std_errors[PRIMARY_MTS], 4),
                      "p": round(p, 4), "N": int(res.nobs)})

sec5_df = pd.DataFrame(sec5_rows)
print()
print(sec5_df.to_string(index=False))


def _direction(b, p, alpha=0.10):
    if p < alpha:
        return f"{'UP' if b > 0 else 'DOWN'} (β={b:+.3f}, p={p:.3f})"
    return f"FLAT (β={b:+.3f}, p={p:.3f})"


bt, pt = sec5_stats["part_n_territorial"]
be, pe = sec5_stats["part_n_extraterritorial"]
terr_dir, extra_dir = _direction(bt, pt), _direction(be, pe)
if bt > 0 and pt < 0.10 and be > 0 and pe < 0.10:
    conclusion = ("amplification is additive — states add extraterritorial "
                  "engagement on top of home conflict")
elif bt < 0 and pt < 0.10 and be > 0 and pe < 0.10:
    conclusion = "substitution pattern present after all — territorial down, extraterritorial up"
elif be > 0 and pe < 0.10:
    conclusion = "engagement grows only on the extraterritorial margin"
elif bt > 0 and pt < 0.10:
    conclusion = "engagement grows only on the territorial margin"
else:
    conclusion = "no significant movement on either margin"
sec5_reading = (f"substitution requires territorial DOWN + extraterritorial UP; "
                f"observed: territorial {terr_dir}, extraterritorial {extra_dir} "
                f"→ {conclusion}")
print(f"\n{sec5_reading}")

sec5_df.to_csv(TBL_DIR / "section5_conflict_type_decomposition.csv", index=False)
print(f"\n[Section 5] Conflict-type decomposition done — saved → "
      f"{TBL_DIR / 'section5_conflict_type_decomposition.csv'}")

Rows clipped at 0 in part_n_territorial derivation: 0



                Outcome  β_mts     SE      p    N
     part_n_territorial 0.6658 0.1938 0.0006 4896
part_n_extraterritorial 0.1602 0.2018 0.4272 4896

substitution requires territorial DOWN + extraterritorial UP; observed: territorial UP (β=+0.666, p=0.001), extraterritorial FLAT (β=+0.160, p=0.427) → engagement grows only on the territorial margin

[Section 5] Conflict-type decomposition done — saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb11\section5_conflict_type_decomposition.csv


## Section 5b — Lagged MTS (Reverse-Causality Robustness)

The territorial margin is where reverse causality is most plausible: states already at war raise spending and imports. MTS enters the Section 5 models contemporaneously; here it is replaced with its t−1 and t−3 lags (one timing per fit — contemporaneous MTS and its lags are highly collinear and are never included in the same model). If the lagged β holds, capability *precedes* intensity; if it collapses, the association is contemporaneous/reactive and the paper's causal language must stay associational.

In [8]:
rq1 = rq1.sort_values(["iso3", "year"]).reset_index(drop=True)
for L in (1, 3):
    rq1[f"{PRIMARY_MTS}_lag{L}"] = rq1.groupby("iso3")[PRIMARY_MTS].shift(L)

TIMINGS = {"contemporaneous": PRIMARY_MTS,
           "t-1": f"{PRIMARY_MTS}_lag1",
           "t-3": f"{PRIMARY_MTS}_lag3"}
SEC5B_OUTCOMES = ["log_brd", "part_n_territorial", "part_n_minor"]

sec5b_rows = []
sec5b_stats = {}
for outcome in SEC5B_OUTCOMES:
    for timing, mts_col in TIMINGS.items():
        res, _ = run_panel_fe(rq1, outcome, mts_col, CONTROLS)
        b, se, p = res.params[mts_col], res.std_errors[mts_col], res.pvalues[mts_col]
        sec5b_stats[(outcome, timing)] = (b, se, p)
        sec5b_rows.append({"Outcome": outcome, "timing": timing, "mts_col": mts_col,
                           "β": round(b, 4), "SE": round(se, 4),
                           "p": round(p, 4), "N": int(res.nobs)})

sec5b_df = pd.DataFrame(sec5b_rows)
sec5b_df.to_csv(TBL_DIR / "section5b_lagged_mts.csv", index=False)

print("=== MTS timing grid — β (p) ===\n")
print(f"{'Outcome':<22s} {'contemporaneous':>18s} {'t-1':>18s} {'t-3':>18s}")
for outcome in SEC5B_OUTCOMES:
    cells = [f"{sec5b_stats[(outcome, t)][0]:+.3f} (p={sec5b_stats[(outcome, t)][2]:.3f})"
             for t in TIMINGS]
    print(f"{outcome:<22s} {cells[0]:>18s} {cells[1]:>18s} {cells[2]:>18s}")

print()
sec5b_verdicts = {}
for outcome in SEC5B_OUTCOMES:
    b0, _, p0 = sec5b_stats[(outcome, "contemporaneous")]
    b1, _, p1 = sec5b_stats[(outcome, "t-1")]
    b3, _, p3 = sec5b_stats[(outcome, "t-3")]
    lag1_ok = (b1 > 0) and (p1 < 0.10)
    lag3_ok = (b3 > 0) and (p3 < 0.10)
    if lag1_ok and lag3_ok:
        v = "PRECEDES — capability leads intensity at t-1 and t-3"
    elif lag1_ok:
        drop_pct = 100 * (b0 - b3) / b0 if b0 != 0 else np.nan
        v = (f"ATTENUATES — t-1 holds, t-3 does not "
             f"(β drops {drop_pct:.0f}% from contemporaneous to t-3)")
    elif lag3_ok:
        v = "MIXED — t-3 holds but t-1 does not (interpret with caution)"
    else:
        v = ("CONTEMPORANEOUS ONLY — causal language must remain associational; "
             "capability co-moves with, and may respond to, domestic conflict")
    sec5b_verdicts[outcome] = v
    print(f"{outcome:<22s} {v}")

print(f"\n[Section 5b] {len(sec5b_df)} lagged-MTS fits — saved → "
      f"{TBL_DIR / 'section5b_lagged_mts.csv'}")

=== MTS timing grid — β (p) ===

Outcome                   contemporaneous                t-1                t-3
log_brd                  +2.030 (p=0.001)   +0.970 (p=0.031)   -0.007 (p=0.990)
part_n_territorial       +0.666 (p=0.001)   +0.328 (p=0.039)   +0.148 (p=0.461)
part_n_minor             +1.661 (p=0.008)   +1.494 (p=0.027)   +1.218 (p=0.079)

log_brd                ATTENUATES — t-1 holds, t-3 does not (β drops 100% from contemporaneous to t-3)
part_n_territorial     ATTENUATES — t-1 holds, t-3 does not (β drops 78% from contemporaneous to t-3)
part_n_minor           PRECEDES — capability leads intensity at t-1 and t-3

[Section 5b] 9 lagged-MTS fits — saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb11\section5b_lagged_mts.csv


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
timing_order = list(TIMINGS)
for ax, outcome in zip(axes, SEC5B_OUTCOMES):
    y_pos = np.arange(len(timing_order))
    betas = np.array([sec5b_stats[(outcome, t)][0] for t in timing_order])
    ses = np.array([sec5b_stats[(outcome, t)][1] for t in timing_order])
    ax.errorbar(betas, y_pos, xerr=1.96 * ses,
                fmt="o", capsize=4, color="steelblue")
    ax.axvline(0, color="gray", linestyle="--", lw=0.8)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(timing_order, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(outcome, fontsize=10)
    ax.set_xlabel("β (MTS)", fontsize=9)
fig.suptitle("MTS timing — contemporaneous vs lagged capability (95% CI)", y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig3_lagged_mts.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"[Section 5b] Figure saved → {FIG_DIR / 'fig3_lagged_mts.png'}")

[Section 5b] Figure saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\figures\nb11\fig3_lagged_mts.png


## Section 6 — Robustness Across MTS Variants

The two decisive quantities re-estimated with `mts_milex` and `mts_tiv` (`log_brd`
only): (a) the continuous polyarchy interaction, (b) the M3 Africa marginal
effect.

**Reading the grid:** sign disagreement among *insignificant* estimates supports the null — noise has no stable sign; it is not evidence of fragility. An all-insignificant row is therefore labeled “ROBUSTLY NULL across variants” rather than “SIGN DIVERGES”.

In [10]:
def _star(p):
    return "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.10 else ""))


sec6_rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for mts_var in MTS_VARIANTS:
        res_a, ic_a = run_panel_fe(rq1, "log_brd", mts_var, POLY_CONTROLS,
                                   interaction=POLY)
        res_b, ints_b = run_panel_fe_interactions(rq1, "log_brd", mts_var,
                                                  CONTROLS_M, mods_m3)
        sec6_rows += [
            {"MTS": mts_var, "quantity": "mts × polyarchy interaction",
             "β": round(res_a.params[ic_a], 4),
             "p": round(res_a.pvalues[ic_a], 4),
             "sig": _star(res_a.pvalues[ic_a]), "N": int(res_a.nobs)},
            {"MTS": mts_var, "quantity": "M3 Africa marginal (β_mts)",
             "β": round(res_b.params[mts_var], 4),
             "p": round(res_b.pvalues[mts_var], 4),
             "sig": _star(res_b.pvalues[mts_var]), "N": int(res_b.nobs)},
        ]

sec6_df = pd.DataFrame(sec6_rows)
print(sec6_df.to_string(index=False))

print("\n=== Sign/significance grid (log_brd) ===\n")
print(f"{'Quantity':<32s} {'pca_3feat':>12s} {'milex':>12s} {'tiv':>12s}   verdict")
sec6_verdicts = {}
for quantity in ["mts × polyarchy interaction", "M3 Africa marginal (β_mts)"]:
    cells, betas, ps = [], [], []
    for m in MTS_VARIANTS:
        row = sec6_df.loc[(sec6_df["MTS"] == m)
                          & (sec6_df["quantity"] == quantity)].iloc[0]
        cells.append(f"{'+' if row['β'] > 0 else '-'}{row['sig'] or 'ns'}")
        betas.append(row["β"]); ps.append(row["p"])
    same_sign = len({np.sign(b) for b in betas}) == 1
    all_sig = all(p < 0.10 for p in ps)
    all_null = all(p >= 0.10 for p in ps)
    if all_null:
        verdict = "ROBUSTLY NULL across variants"
    elif same_sign and all_sig:
        verdict = "CONSISTENT"
    elif same_sign:
        verdict = "same sign, significance varies"
    else:
        verdict = "SIGN DIVERGES"
    sec6_verdicts[quantity] = verdict
    print(f"{quantity:<32s} {cells[0]:>12s} {cells[1]:>12s} {cells[2]:>12s}   {verdict}")

sec6_df.to_csv(TBL_DIR / "section6_robustness.csv", index=False)
print(f"\n[Section 6] Variant robustness done — saved → "
      f"{TBL_DIR / 'section6_robustness.csv'}")

          MTS                    quantity       β      p sig    N
mts_pca_3feat mts × polyarchy interaction -0.9550 0.4974     4896
mts_pca_3feat  M3 Africa marginal (β_mts)  4.5340 0.0006 *** 4896
    mts_milex mts × polyarchy interaction  0.6981 0.6930     4896
    mts_milex  M3 Africa marginal (β_mts)  2.5779 0.1369     4896
      mts_tiv mts × polyarchy interaction  0.0533 0.9282     4809
      mts_tiv  M3 Africa marginal (β_mts)  1.3588 0.0051 *** 4809

=== Sign/significance grid (log_brd) ===

Quantity                            pca_3feat        milex          tiv   verdict
mts × polyarchy interaction               -ns          +ns          +ns   ROBUSTLY NULL across variants
M3 Africa marginal (β_mts)               +***          +ns         +***   same sign, significance varies

[Section 6] Variant robustness done — saved → D:\post graduate\assignments and projects\sem 3\BDA\PeaceMakers' Arms Race\version 2\tables\nb11\section6_robustness.csv


## Section 7 — Sanity Checks

In [11]:
checks = []

# [1] LOO jackknife coverage
checks.append((f"LOO jackknife ≥150 successful fits ({len(loo_df)}), "
               f"failures: {len(loo_failures)}",
               len(loo_df) >= 150 and len(loo_failures) == 0))

# [2] Section 2 refit count = African countries with non-zero log_brd + 1 joint
n_refits = len(sec2_df) - 1  # exclude the "(none)" reference row
checks.append((f"Africa jackknife refits ({n_refits}) == countries with "
               f"non-zero log_brd ({len(afr_counts)}) + 1 joint",
               n_refits == len(afr_counts) + 1))

# [3] polyarchy interaction models converged for both outcomes
checks.append(("Polyarchy interaction converged (MTS main effect present, both outcomes)",
               all(PRIMARY_MTS in sec3_results[o][0].params.index
                   for o in SEC3_OUTCOMES)))

# [4] M1/M2/M3 converged; M3 contains both interaction sets
ok4 = True
for outcome in SEC4_OUTCOMES:
    m1, m2, m3, m3_ints = sec4_models[outcome]
    region_ints_present = all(c in m3.params.index for lbl, c in m3_ints.items()
                              if lbl != "polyarchy")
    poly_int_present = m3_ints["polyarchy"] in m3.params.index
    ok4 = ok4 and (PRIMARY_MTS in m1.params.index) and (PRIMARY_MTS in m2.params.index) \
              and region_ints_present and poly_int_present
checks.append(("M1/M2/M3 converged; M3 holds region AND polyarchy interactions", ok4))

# [5] territorial derivation clipped 0 rows
checks.append((f"part_n_territorial clipped rows == 0 (got {n_clip})", n_clip == 0))

# [6] all tables and figures saved
expected_tables = [
    "section1_loo_jackknife.csv", "section2_africa_jackknife.csv",
    "section3_polyarchy_interaction.csv", "section4_region_vs_polyarchy.csv",
    "section5_conflict_type_decomposition.csv", "section6_robustness.csv",
]
expected_figs = ["fig1_loo_distribution.png", "fig2_marginal_by_polyarchy.png"]
missing = ([t for t in expected_tables if not (TBL_DIR / t).exists()]
           + [f for f in expected_figs if not (FIG_DIR / f).exists()])
checks.append((f"All 6 tables + 2 figures saved (missing: {missing or 'none'})",
               len(missing) == 0))

# [7] lagged-MTS table exists with 9 fits
checks.append((f"section5b_lagged_mts.csv exists with 9 fits (got {len(sec5b_df)})",
               (TBL_DIR / "section5b_lagged_mts.csv").exists() and len(sec5b_df) == 9))

# [8] Section 5b figure saved
checks.append(("Figure fig3_lagged_mts.png saved",
               (FIG_DIR / "fig3_lagged_mts.png").exists()))

# [9] lag columns built with plausible non-null counts
n_c = int(rq1[PRIMARY_MTS].notna().sum())
n_l1 = int(rq1[f"{PRIMARY_MTS}_lag1"].notna().sum())
n_l3 = int(rq1[f"{PRIMARY_MTS}_lag3"].notna().sum())
checks.append((f"MTS lag columns built (non-null: contemp {n_c:,} > lag1 {n_l1:,} > lag3 {n_l3:,})",
               n_l1 < n_c and n_l3 < n_l1))

# [10] Section 2 coverage flag computed
checks.append((f"Section 2 coverage flag computed: not_in_sample={not_in_sample}",
               isinstance(not_in_sample, list)))

print("=== NB11 sanity checks ===\n")
n_pass = 0
for i, (label, ok) in enumerate(checks, 1):
    status = "PASS" if ok else "FAIL"
    n_pass += int(ok)
    print(f"[{i}] {status} — {label}")
print(f"\n{n_pass}/{len(checks)} checks passed")

=== NB11 sanity checks ===

[1] PASS — LOO jackknife ≥150 successful fits (192), failures: 0
[2] PASS — Africa jackknife refits (35) == countries with non-zero log_brd (34) + 1 joint
[3] PASS — Polyarchy interaction converged (MTS main effect present, both outcomes)
[4] PASS — M1/M2/M3 converged; M3 holds region AND polyarchy interactions
[5] PASS — part_n_territorial clipped rows == 0 (got 0)
[6] PASS — All 6 tables + 2 figures saved (missing: none)
[7] PASS — section5b_lagged_mts.csv exists with 9 fits (got 9)
[8] PASS — Figure fig3_lagged_mts.png saved
[9] PASS — MTS lag columns built (non-null: contemp 5,097 > lag1 4,950 > lag3 4,651)
[10] PASS — Section 2 coverage flag computed: not_in_sample=['SOM', 'ERI', 'COM']

10/10 checks passed


## Section 8 — Headline Findings

In [12]:
print("=" * 74)
print("NB11 HEADLINE FINDINGS — mechanisms behind amplification and Africa effect")
print("=" * 74)

print("\n1. Amplification systemic or outlier-driven? (Section 1)")
print(f"   Full-sample β={full_b:+.3f}; LOO range "
      f"[{min_row['β']:+.3f} (drop {min_row['iso3']}), "
      f"{max_row['β']:+.3f} (drop {max_row['iso3']})]; "
      f"drop-top-5 β={drop5_b:+.3f} (p={drop5_p:.3f})")
print(f"   → {sec1_verdict}")

print("\n2. Africa effect broad or few-states? (Section 2)")
print(f"   → {sec2_verdict}")
print(f"   Coverage caveat: {not_in_sample} have non-zero log_brd but no usable "
      f"MTS/controls — outside the estimation sample.")

print("\n3. Does regime type moderate amplification? (Section 3)")
for outcome in SEC3_OUTCOMES:
    res, int_col = sec3_results[outcome]
    b02 = marginal_at(res, PRIMARY_MTS, int_col, 0.2)
    b08 = marginal_at(res, PRIMARY_MTS, int_col, 0.8)
    print(f"   {outcome:<15s} β@0.2={b02[0]:+.3f} (p={b02[2]:.3f}) vs "
          f"β@0.8={b08[0]:+.3f} (p={b08[2]:.3f}), "
          f"interaction p={res.pvalues[int_col]:.3f} → {sec3_reading[outcome]}")

print("\n4. Is 'Africa' really 'autocracy'? (Section 4, M1 vs M3)")
for outcome in SEC4_OUTCOMES:
    print(f"   {outcome:<15s} → {sec4_verdicts[outcome]}")

print("\n5. Why did substitution fail? (Section 5)")
print(f"   → {sec5_reading}")

print("\n6. Which findings survive all three MTS variants? (Section 6, log_brd)")
for quantity, verdict in sec6_verdicts.items():
    print(f"   {quantity:<32s} → {verdict}")

print("\n7. Does capability PRECEDE intensity? (Section 5b)")
for outcome in SEC5B_OUTCOMES:
    print(f"   {outcome:<22s} → {sec5b_verdicts[outcome]}")
b_line = ", ".join(f"{t}: {sec5b_stats[('log_brd', t)][0]:+.3f} "
                   f"(p={sec5b_stats[('log_brd', t)][2]:.3f})" for t in TIMINGS)
print(f"   log_brd β by timing — {b_line}")

print()
print("=== NB-11 complete — proceed to NB-12 (RQ2 power analysis) ===")

NB11 HEADLINE FINDINGS — mechanisms behind amplification and Africa effect

1. Amplification systemic or outlier-driven? (Section 1)
   Full-sample β=+2.030; LOO range [+1.819 (drop LKA), +2.157 (drop KWT)]; drop-top-5 β=+2.025 (p=0.001)
   → SYSTEMIC — every LOO β stays positive with p<0.05; drop-top-5 also survives

2. Africa effect broad or few-states? (Section 2)
   → BROAD — Africa β stays positive with p<0.10 in every single-country drop and the joint top-3 drop
   Coverage caveat: ['SOM', 'ERI', 'COM'] have non-zero log_brd but no usable MTS/controls — outside the estimation sample.

3. Does regime type moderate amplification? (Section 3)
   part_n_minor    β@0.2=+1.465 (p=0.111) vs β@0.8=+1.930 (p=0.108), interaction p=0.784 → no significant regime-type moderation
   log_brd         β@0.2=+2.274 (p=0.003) vs β@0.8=+1.701 (p=0.012), interaction p=0.497 → no significant regime-type moderation

4. Is 'Africa' really 'autocracy'? (Section 4, M1 vs M3)
   log_brd         → region ef